# Lab 13 — Vision Transformers: ViT e DeiT-Tiny

## Objetivos
1. Explorar ViT pré-treinado com `timm` (inferência zero-shot no ImageNet)
2. Fine-tuning do DeiT-Tiny no CIFAR-10 com HuggingFace Transformers
3. Comparar com CNN do zero (Lab 10) e Transfer Learning CNN (Lab 12)
4. Salvar artefato para deploy via FastAPI

---
**GPU recomendada** — DeiT-Tiny é leve (~5.7M params) e roda em CPU, mas GPU acelera o treino ~5×.

In [ ]:
# %pip install transformers timm accelerate datasets torch torchvision pillow matplotlib

## Parte 1 — ViT pré-treinado com timm (ImageNet)

In [ ]:
import timm
import torch
from PIL import Image
import urllib.request
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

# Listar modelos ViT disponíveis no timm
vit_models = timm.list_models('vit*', pretrained=True)
print(f'Modelos ViT disponíveis: {len(vit_models)}')
print('Exemplos:', vit_models[:5])

In [ ]:
# Carregar DeiT-Tiny pré-treinado via timm
model_timm = timm.create_model('deit_tiny_patch16_224', pretrained=True)
model_timm.eval().to(device)

print(f'Parâmetros: {sum(p.numel() for p in model_timm.parameters()):,}')

# Pré-processamento padrão do timm
data_config = timm.data.resolve_model_data_config(model_timm)
transforms = timm.data.create_transform(**data_config, is_training=False)
print('\nTransforms:', transforms)

In [ ]:
# Inferência com imagem de teste
urllib.request.urlretrieve(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg',
    'teste.jpg'
)

img = Image.open('teste.jpg').convert('RGB')
tensor = transforms(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model_timm(tensor)
    probs = torch.softmax(logits, dim=1)[0]

top5_probs, top5_idx = torch.topk(probs, 5)

# Rótulos do ImageNet
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt',
    'imagenet_classes.txt'
)
with open('imagenet_classes.txt') as f:
    imagenet_classes = [l.strip() for l in f]

plt.figure(figsize=(4, 4))
plt.imshow(img); plt.axis('off'); plt.title('Imagem de teste'); plt.show()

print('Top-5 previsões (DeiT-Tiny, ImageNet):')
for i, p in zip(top5_idx, top5_probs):
    print(f'  {imagenet_classes[i]:30s}  {p*100:.1f}%')

## Parte 2 — Fine-tuning do DeiT-Tiny no CIFAR-10

In [ ]:
import json
from pathlib import Path
import torchvision
import torchvision.transforms as T
from transformers import AutoImageProcessor, AutoModelForImageClassification

CLASSES = ['aviao', 'automovel', 'passaro', 'gato', 'cervo',
           'cachorro', 'sapo', 'cavalo', 'navio', 'caminhao']
MODEL_ID = 'facebook/deit-tiny-patch16-224'

In [ ]:
# Processor do HuggingFace cuida de resize + normalização
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

def transform_hf(examples):
    """Compatível com HuggingFace datasets (pode adaptar para torchvision)."""
    return processor(images=examples, return_tensors='pt')

# Dataset CIFAR-10 via torchvision com transforms manuais
size = processor.size.get('height', 224)

tf_train = T.Compose([
    T.Resize((size, size)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=processor.image_mean, std=processor.image_std),
])
tf_val = T.Compose([
    T.Resize((size, size)),
    T.ToTensor(),
    T.Normalize(mean=processor.image_mean, std=processor.image_std),
])

trainset = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=tf_train)
testset  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tf_val)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False, num_workers=2)

print(f'Treino: {len(trainset)} | Teste: {len(testset)}')

In [ ]:
# Carrega DeiT-Tiny e substitui o head para 10 classes
model_deit = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=10,
    ignore_mismatched_sizes=True,   # descarta o classifier original (1000 classes)
)
model_deit = model_deit.to(device)

treinaveis = sum(p.numel() for p in model_deit.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_deit.parameters())
print(f'Parâmetros treináveis: {treinaveis:,} / {total:,}')

In [ ]:
import torch.nn as nn

optimizer = torch.optim.AdamW(model_deit.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

historico = {'train_acc': [], 'val_acc': []}

for epoch in range(10):
    model_deit.train()
    corretos = total_b = 0
    for imgs, labels in trainloader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        # HuggingFace espera pixel_values
        outputs = model_deit(pixel_values=imgs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        corretos += (outputs.logits.argmax(1) == labels).sum().item()
        total_b += labels.size(0)

    train_acc = 100 * corretos / total_b

    model_deit.eval()
    c_val = t_val = 0
    with torch.no_grad():
        for imgs, labels in testloader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model_deit(pixel_values=imgs)
            c_val += (out.logits.argmax(1) == labels).sum().item()
            t_val += labels.size(0)

    val_acc = 100 * c_val / t_val
    scheduler.step()

    historico['train_acc'].append(train_acc)
    historico['val_acc'].append(val_acc)
    print(f'Época {epoch+1:02d}/10 | Treino: {train_acc:.1f}% | Val: {val_acc:.1f}%')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historico['train_acc'], label='DeiT-Tiny treino')
plt.plot(historico['val_acc'],   label='DeiT-Tiny validação')
plt.axhline(y=73, color='gray',   linestyle='--', label='CNN do zero (ref.)')
plt.axhline(y=87, color='orange', linestyle='--', label='MobileNetV3 TL (ref.)')
plt.title('DeiT-Tiny Fine-Tuning — CIFAR-10')
plt.xlabel('Época'); plt.ylabel('Acurácia (%)')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Melhor val acc: {max(historico["val_acc"]):.1f}%')

## Parte 3 — Salvar artefato

In [ ]:
artifacts_dir = Path('artifacts/deit_cifar10')
artifacts_dir.mkdir(parents=True, exist_ok=True)

# salva no formato HuggingFace (config + pesos)
model_deit.save_pretrained(str(artifacts_dir))
processor.save_pretrained(str(artifacts_dir))

metadata = {
    'model_name': 'DeiT-Tiny (fine-tuned CIFAR-10)',
    'base_model': MODEL_ID,
    'dataset': 'CIFAR-10',
    'num_classes': 10,
    'classes': CLASSES,
    'val_acc': round(max(historico['val_acc']), 2),
}
Path('artifacts/metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('Artefatos salvos:')
for f in sorted(Path('artifacts').rglob('*')):
    if f.is_file():
        print(f'  {f}  ({f.stat().st_size/1024:.0f} KB)')

---
## Exercícios

1. **Comparativo completo:** monte uma tabela com acurácia final e tempo de treino para CNN (Lab10), MobileNetV3 (Lab12) e DeiT-Tiny (Lab13).

2. **Swin-Tiny:** substitua `deit-tiny` por `microsoft/swin-tiny-patch4-window7-224`. O que muda na arquitetura? Como os resultados se comparam?

3. **Mapa de atenção:** visualize os mapas de atenção do DeiT para entender em quais regiões da imagem o modelo presta atenção.

4. **CP1 — preview:** você vai usar CNN (Lab12) ou ViT (Lab13) para o seu checkpoint? Justifique a escolha considerando o dataset que pretende usar.